# Understanding Transformer Circuits Across Languages

## A Complete Guide: From Transformer Architecture to Cross-Lingual Mechanistic Interpretability

This notebook walks through everything from the ground up:
1. **Part 1: How Transformers Work** — the architecture, weight matrices, and how prediction happens
2. **Part 2: The Task** — subject-verb agreement and why it's perfect for interpretability
3. **Part 3: The Experiments** — every experiment in this project, with code, math, and intuition
4. **Part 4: Cross-Lingual Analysis** — extending to 7 languages and what we found
5. **Part 5: Cross-Model Comparison** — do Gemma 2B and BLOOM 3B use the same circuits?

By the end, you'll understand every weight matrix, every hook, and every experiment in the `circuits/` codebase.

---

# Part 0: The Foundations — Vectors, Dimensions, and Why the Numbers Are What They Are

## 0.1 What Is a Vector, Really?

You remember a vector has **magnitude** (how long it is) and **direction** (where it points). In physics class, that's an arrow in 2D or 3D space:

```
2D vector: [3, 4]     → points right 3, up 4, magnitude = 5
3D vector: [1, 0, 2]  → points right 1, forward 0, up 2
```

But there's nothing stopping us from having more numbers. A **2048-dimensional vector** is just a list of 2048 numbers:

```
[0.12, -0.87, 0.33, 1.05, -0.44, ..., 0.72]    ← 2048 numbers
```

We can't visualize 2048 dimensions, but all the same math works:
- **Magnitude** = sqrt(sum of squares) = sqrt(0.12² + (-0.87)² + ... + 0.72²)
- **Direction** = the pattern of which numbers are big/small/positive/negative
- **Dot product** = element-wise multiply and sum (measures similarity)

### What do the 2048 numbers "mean"?

Nobody assigned meanings to them. The model **learned** what to put in each dimension during training. But we can think of each dimension as encoding some feature of the text. Some dimensions might encode:
- Dimension 47: "how noun-like is this token" 
- Dimension 923: "is this token part of a plural subject"
- Dimension 1456: "is this token near the end of a sentence"

In practice, meaning is spread across many dimensions in complex patterns (not one-per-dimension), but the intuition holds: **the 2048 numbers together encode everything the model knows about a token at a given point in processing.**

## 0.2 Why 2048? Why 256? Why 16384? — The Dimension Budget

The model has a **dimension budget** — a fixed-width pipe that ALL information flows through. Think of it like a highway:

```
d_model = 2048 lanes wide
════════════════════════════════════════════════════════════

Every token's representation is a 2048-lane highway.
All information — meaning, grammar, position, context — must fit in these 2048 lanes.
This is the RESIDUAL STREAM.
```

Now, attention heads and MLPs need to **read from** and **write to** this 2048-lane highway. But they don't each get the full 2048 dimensions to work with internally.

### Attention heads: d_head = 256

Each attention head works in a **compressed subspace** of 256 dimensions. Why? Because there are 8 heads per layer, and they need to share the 2048-dimensional highway:

```
d_model = 2048 total dimensions
÷ 8 heads per layer
= 256 dimensions per head (d_head)
```

Think of each head as getting a **private office** with 256 desks. It reads from the 2048-lane highway (compressing 2048 → 256 via W_V), does its work in its 256-dim office, and writes the result back to the highway (expanding 256 → 2048 via W_O).

```
Highway (2048 dims)
    │
    ├──→ Head 0: reads 2048, compresses to 256, works, writes back 2048
    ├──→ Head 1: reads 2048, compresses to 256, works, writes back 2048
    ├──→ Head 2: reads 2048, compresses to 256, works, writes back 2048
    │    ...
    ├──→ Head 7: reads 2048, compresses to 256, works, writes back 2048
    │
    ▼
Highway (2048 dims) — now updated with all 8 heads' contributions
```

Each head sees the FULL highway (2048 dims) as input, but processes through a 256-dim bottleneck. This bottleneck forces each head to focus on a specific aspect of the data — it can't just copy everything through.

### MLP: d_mlp = 16384

The MLP is the opposite — it **expands** to a much wider space:

```
Highway (2048 dims)
    │
    ├──→ MLP: expands 2048 → 16384, processes, compresses back to 2048
    │
    ▼
Highway (2048 dims) — updated with MLP's contribution
```

Why 16384? That's **8x wider** than the highway. The MLP needs this extra room because it does **nonlinear** computation (applying activation functions, combining features). More neurons = more capacity for complex feature detection.

Think of it like this:
- The **highway (2048)** is for transporting information between layers
- The **attention heads (256 each)** are narrow specialists — each one extracts one specific pattern
- The **MLP (16384)** is a wide processing factory — it combines many features in complex ways

## 0.3 What Are Weight Matrices? — The Machines That Transform Vectors

A **weight matrix** is a grid of numbers that transforms one vector into another. It's the "machine" that does the actual computation.

### A simple example (2D → 3D)

```
Input vector:  [3, 4]  (2 dimensions)

Weight matrix W (3 rows × 2 columns):
    ┌         ┐
    │ 0.5  0.1│
    │ 0.0  0.8│
    │-0.3  0.2│
    └         ┘

Output = input @ W^T:
    output[0] = 3×0.5 + 4×0.1 = 1.9
    output[1] = 3×0.0 + 4×0.8 = 3.2
    output[2] = 3×(-0.3) + 4×0.2 = -0.1
    
Output vector: [1.9, 3.2, -0.1]  (3 dimensions!)
```

The matrix took a 2D vector and produced a 3D vector. That's what weight matrices do — **they change the dimensionality** and **they change the direction** of vectors.

### In the transformer:

| Weight Matrix | Input dims → Output dims | What it does |
|--------------|-------------------------|--------------|
| **W_V** | 2048 → 256 | Compresses from highway to head's office |
| **W_O** | 256 → 2048 | Expands from head's office back to highway |
| **W_Q** | 2048 → 256 | Creates the "query" (what am I looking for?) |
| **W_K** | 2048 → 256 | Creates the "key" (what do I contain?) |
| **W_in** | 2048 → 16384 | Expands from highway to MLP's wide space |
| **W_out** | 16384 → 2048 | Compresses from MLP back to highway |
| **W_U** | 2048 → 256128 | Maps to vocabulary (one score per word!) |

### How many numbers is that?

Each weight matrix is a grid. W_V for one head is 2048×256 = **524,288 numbers**. The whole model has about **2 billion** of these numbers (hence "Gemma **2B**"). These numbers are what the model "learned" during training — they encode everything it knows about language.

When we say "weight-level analysis" (Experiment 7), we mean: looking at these actual numbers in the matrices and asking which ones are important for the SVA task.

## 0.4 The Dot Product — How the Model Measures Similarity

One operation shows up everywhere: the **dot product**. It measures how much two vectors point in the same direction.

```
a = [1, 0, 3]
b = [2, 0, 1]

dot product = 1×2 + 0×0 + 3×1 = 5    (positive = same direction)

c = [1, 0, -3]  
a · c = 1×1 + 0×0 + 3×(-3) = -8       (negative = opposite direction)

d = [0, 5, 0]
a · d = 1×0 + 0×5 + 3×0 = 0           (zero = perpendicular)
```

This is THE core operation in transformers:
- **Attention scores** = query · key (do these positions match?)
- **DLA** = component_output · unembed_direction (does this component help predict the right verb?)
- **Logit diff** = residual_stream · unembed_direction (does the model prefer "is" or "are"?)

Every time we ask "does X align with Y?", we're computing a dot product.

---

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, ax = plt.subplots(figsize=(16, 8))
ax.set_xlim(0, 16)
ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title("How the dimensions fit together in one layer of Gemma 2B", fontsize=14, pad=20)

# Residual stream (highway)
highway_y = 8.5
ax.add_patch(mpatches.FancyBboxPatch((0.5, highway_y-0.3), 15, 0.6, 
             boxstyle="round,pad=0.1", facecolor="#3498db", alpha=0.3))
ax.text(8, highway_y, "RESIDUAL STREAM (d_model = 2048 dimensions)", 
        ha="center", va="center", fontsize=11, fontweight="bold")

# Attention heads
head_y = 5.5
ax.text(4.5, 7.5, "ATTENTION LAYER", ha="center", fontsize=10, fontweight="bold", color="#555")
for i in range(8):
    x = 0.8 + i * 1.05
    color = "#e74c3c" if i == 7 else "#e74c3c80"
    ax.add_patch(mpatches.FancyBboxPatch((x, head_y-0.4), 0.85, 0.8,
                 boxstyle="round,pad=0.05", facecolor=color, alpha=0.6))
    label = f"H{i}" if i != 7 else "H7*"
    ax.text(x+0.42, head_y, label, ha="center", va="center", fontsize=8, fontweight="bold")

ax.text(4.5, head_y-0.7, "8 heads × 256 dims each = 2048 total", ha="center", fontsize=9, color="#666")

# Arrows: highway → heads → highway
for i in range(8):
    x = 0.8 + i * 1.05 + 0.42
    ax.annotate("", xy=(x, head_y+0.4), xytext=(x, highway_y-0.3),
                arrowprops=dict(arrowstyle="->", color="#999", lw=0.8))

# W_V and W_O labels
ax.text(0.3, 6.8, "W_V\n2048→256\n(compress)", fontsize=7, ha="center", color="#e74c3c")
ax.text(0.3, 5.0, "W_O\n256→2048\n(expand)", fontsize=7, ha="center", color="#e74c3c")

# MLP block
mlp_y = 2.5
ax.text(12, 7.5, "MLP LAYER", ha="center", fontsize=10, fontweight="bold", color="#555")
ax.add_patch(mpatches.FancyBboxPatch((9.5, mlp_y-0.5), 5, 1.0,
             boxstyle="round,pad=0.1", facecolor="#4ecdc4", alpha=0.4))
ax.text(12, mlp_y, "MLP: 16,384 neurons", ha="center", va="center", fontsize=10, fontweight="bold")
ax.text(12, mlp_y-0.8, "8× wider than the highway!", ha="center", fontsize=9, color="#666")

# Arrows for MLP
ax.annotate("", xy=(12, mlp_y+0.5), xytext=(12, highway_y-0.3),
            arrowprops=dict(arrowstyle="->", color="#999", lw=1.5))
ax.text(12.8, 5.8, "W_in: 2048→16384", fontsize=8, color="#4ecdc4", rotation=90)
ax.text(11.2, 5.8, "W_out: 16384→2048", fontsize=8, color="#4ecdc4", rotation=90)

# Output highway
out_y = 1.0
ax.add_patch(mpatches.FancyBboxPatch((0.5, out_y-0.3), 15, 0.6,
             boxstyle="round,pad=0.1", facecolor="#3498db", alpha=0.3))
ax.text(8, out_y, "RESIDUAL STREAM (still 2048 dims — now updated)", 
        ha="center", va="center", fontsize=11, fontweight="bold")

# Summary box
ax.add_patch(mpatches.FancyBboxPatch((0.5, -0.2), 15, 0.8,
             boxstyle="round,pad=0.1", facecolor="#f9f9f9", edgecolor="#ccc"))
ax.text(8, 0.2, "The highway is always 2048. Heads compress to 256, MLPs expand to 16384, but everything goes back to 2048.",
        ha="center", va="center", fontsize=10, style="italic")

# Arrows down
for i in range(8):
    x = 0.8 + i * 1.05 + 0.42
    ax.annotate("", xy=(x, out_y+0.3), xytext=(x, head_y-0.4),
                arrowprops=dict(arrowstyle="->", color="#999", lw=0.8))

ax.annotate("", xy=(12, out_y+0.3), xytext=(12, mlp_y-0.5),
            arrowprops=dict(arrowstyle="->", color="#999", lw=1.5))

plt.tight_layout()
plt.show()

# Print concrete numbers
print("Dimension summary for Gemma 2B:")
print(f"  Residual stream:  {2048:>6} dims  (the highway)")
print(f"  Per head:         {256:>6} dims  (2048 ÷ 8 heads)")
print(f"  MLP neurons:      {16384:>6} dims  (2048 × 8)")
print(f"  Vocabulary:       {256128:>6} dims  (one per token)")
print()
print("Weight matrix sizes (number of learnable parameters):")
print(f"  W_V (one head):   2048 × 256   = {2048*256:>10,} params")
print(f"  W_O (one head):   256 × 2048   = {256*2048:>10,} params")
print(f"  W_Q (one head):   2048 × 256   = {2048*256:>10,} params")
print(f"  W_K (one head):   2048 × 256   = {2048*256:>10,} params")
print(f"  All 8 heads:      × 8          = {2048*256*4*8:>10,} params")
print(f"  MLP (W_in+W_out): 2048×16384×2 = {2048*16384*2:>10,} params")
print(f"  One full layer:                  {2048*256*4*8 + 2048*16384*2:>10,} params")
print(f"  18 layers:                       {(2048*256*4*8 + 2048*16384*2)*18:>10,} params")
print(f"  + embeddings etc ≈               2,000,000,000 params (2B)")

# Part 1: How Transformers Work

## 1.1 The Big Picture

A transformer takes a sequence of words (tokens), processes them through a stack of layers, and outputs a probability distribution over what the next word should be.

```
Input text: "The doctor that helped the teacher"
     |
     v
[Tokenizer] --> token IDs: [651, 7099, 674, 7548, 651, 6573]
     |
     v
[Embedding] --> vectors: each token becomes a vector in R^2048
     |
     v
[Layer 0]  --> attention heads + MLP modify the vectors
[Layer 1]  --> attention heads + MLP modify the vectors
   ...
[Layer 17] --> attention heads + MLP modify the vectors
     |
     v
[Unembedding] --> dot each vector with every word in vocabulary
     |
     v
Output: P("is") = 0.73, P("are") = 0.02, P("was") = 0.05, ...
```

**Gemma 2B** has:
- **18 layers** (numbered 0-17)
- **8 attention heads per layer** (numbered 0-7)
- **d_model = 2048**: every vector in the model lives in 2048-dimensional space
- **d_head = 256**: each attention head works in a 256-dimensional subspace
- **d_mlp = 16384**: each MLP layer has 16,384 neurons
- **vocab_size = 256,128**: the model knows ~256K tokens

## 1.2 The Residual Stream

The most important concept in mechanistic interpretability is the **residual stream**. Think of it as a shared communication bus that every component reads from and writes to.

```
Token embedding (2048 dims)
        |
        v
   ┌────────────── RESIDUAL STREAM ──────────────┐
   │                                              │
   │  x₀ = embedding("The")                      │
   │    │                                         │
   │    ├── Attention Layer 0 reads x₀            │
   │    │   writes back: x₀ += attn_output_0     │
   │    │                                         │
   │    ├── MLP Layer 0 reads x₀                  │
   │    │   writes back: x₀ += mlp_output_0      │
   │    │                                         │
   │    ├── Attention Layer 1 reads x₀            │
   │    │   writes back: x₀ += attn_output_1     │
   │    │                                         │
   │    ...18 layers of this...                   │
   │    │                                         │
   │    v                                         │
   │  x_final = x₀ + Σ(all attention outputs)    │
   │                + Σ(all MLP outputs)          │
   │                                              │
   └──────────────────────────────────────────────┘
        |
        v
   Unembedding → logits → probabilities
```

**Key insight**: The final vector is literally the **sum** of the initial embedding plus every component's output. This additive structure is what makes interpretability possible — we can ask "how much did each component contribute?" and get an exact answer (not an approximation).

In code, TransformerLens lets us access the residual stream at any point:
```python
# hook_resid_post at layer L = residual stream after layer L
cache["blocks.5.hook_resid_post"]  # shape: (batch, seq_len, 2048)
```

## 1.3 Attention Heads: The Four Weight Matrices

Each attention head has **four** weight matrices. Let's go through them one by one.

### W_Q (Query weights) — "What am I looking for?"
- Shape: `(d_model, d_head)` = `(2048, 256)`
- Takes the residual stream vector at the current position and produces a **query**: "I'm looking for information about X"

### W_K (Key weights) — "What do I contain?"
- Shape: `(d_model, d_head)` = `(2048, 256)`
- Takes the residual stream vector at each position and produces a **key**: "I contain information about Y"

### W_V (Value weights) — "What information do I carry?"
- Shape: `(d_model, d_head)` = `(2048, 256)`
- Takes the residual stream vector at attended positions and produces a **value**: "Here's the actual information to transfer"

### W_O (Output weights) — "How do I write back?"
- Shape: `(d_head, d_model)` = `(256, 2048)`
- Takes the value vector (256 dims) and projects it back to the residual stream (2048 dims)

### How they work together:

```
Step 1: ATTENTION PATTERN (which positions to look at)
   For each position i, compute:
      q_i = residual_stream[i] @ W_Q     → (256,)  "what am I looking for?"
      k_j = residual_stream[j] @ W_K     → (256,)  "what does position j have?"
      
      attention_score[i,j] = (q_i · k_j) / sqrt(256)
      attention_weights = softmax(attention_scores)   → sums to 1.0

Step 2: VALUE EXTRACTION (what information to move)
      v_j = residual_stream[j] @ W_V     → (256,)  "information at position j"
      
      z_i = Σ_j attention_weights[i,j] * v_j   → (256,)  "weighted sum of values"
      This is the "hook_z" activation in TransformerLens!

Step 3: OUTPUT PROJECTION (write back to residual stream)
      output_i = z_i @ W_O               → (2048,)  "projected back to model space"
      residual_stream[i] += output_i      → added to the stream
```

### Concrete Example

When the model processes "The doctor that helped the teacher ___":

At the final position (where we predict the verb), **L13H7** does:
1. Its **query** says: "I'm looking for the grammatical subject"
2. Its **keys** at positions 1-2 ("doctor") have high match → high attention
3. Its **values** at "doctor" carry the information "this is singular"
4. Its **output** writes "singular subject detected" back to the residual stream
5. Later layers use this to predict "is" instead of "are"

## 1.4 The OV Circuit and the QK Circuit

There's a cleaner way to think about what an attention head does. It really only has **two** functional circuits:

### The QK Circuit: "WHERE to look"
```
QK = W_Q @ W_K^T     shape: (d_model, d_model)

attention_score[i,j] = residual[i] @ QK @ residual[j]^T / sqrt(d_head)
```
The QK matrix determines the attention pattern — which positions attend to which. It's a bilinear form on the residual stream: "does the content at position i want to look at the content at position j?"

For SVA, the QK circuit of L13H7 makes the final position attend to the subject noun. The query at the verb position says "where is the subject?" and the key at the subject position says "I'm the subject."

### The OV Circuit: "WHAT to write"
```
OV = W_V @ W_O     shape: (d_model, d_model)

output[i] = Σ_j attention[i,j] * (residual[j] @ OV)
```
The OV matrix determines what information the head moves. It reads from the residual stream (via W_V), processes it in d_head space, and writes back (via W_O). It's a (d_model → d_model) linear map that passes through a d_head bottleneck.

For SVA, the OV circuit of L13H7 reads the subject noun's embedding and writes back "this is singular" or "this is plural" — a signal that later layers use to choose the verb.

### Why this decomposition matters

The QK and OV circuits are **independent** — you can analyze them separately:
- QK tells you the head's attention pattern (structural: where info flows)
- OV tells you the head's transformation (functional: what info gets written)

This separation is the foundation of the weight-level analysis in `circuit_map.py`. We can ask "is this head's OV circuit aligned with the verb-number direction?" without ever running data through the model.

## 1.5 MLP Blocks

After each attention layer, there's an MLP (multi-layer perceptron) block. In Gemma 2B, it uses a **gated architecture**:

```
Input: x (from residual stream, 2048 dims)

gate   = x @ W_gate     → (16384,)    "which neurons to activate"
hidden = x @ W_in       → (16384,)    "raw neuron values"
post   = GELU(gate) * hidden           "gated activation" (element-wise multiply)
output = post @ W_out   → (2048,)     "project back to residual stream"

residual_stream += output
```

Each MLP has **16,384 neurons**. Each neuron:
1. Reads a direction from the residual stream (one row of W_in)
2. Gets gated by another direction (one row of W_gate)  
3. Writes back a direction to the residual stream (one row of W_out)

### The "post" activation (hook_post)

The `hook_post` activation is the gated output — `GELU(gate) * hidden` — with shape `(d_mlp,)`. This is what we analyze in the neuron experiment. Each element is one neuron's activation value.

A single neuron's contribution to the logit difference is:
```
neuron_DLA[i] = post[i] * (W_out[i, :] @ unembed_dir)
                ^^^^^^^^   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                how active   how aligned with the task
                it is        direction
```

This decomposes the MLP's total contribution into individual neuron contributions — exactly.

## 1.6 The Unembedding Matrix and How Prediction Works

At the very end, the model needs to convert its 2048-dimensional residual stream vector into a prediction over ~256K vocabulary tokens.

### W_U: The Unembedding Matrix
- Shape: `(d_model, vocab_size)` = `(2048, 256128)`
- Each **column** of W_U is a 2048-dimensional vector representing one token in vocabulary space

### How the model predicts:
```
final_residual = x₀ + Σ(all attention outputs) + Σ(all MLP outputs)    → (2048,)

logits = final_residual @ W_U    → (256128,)     one score per vocab token

probabilities = softmax(logits)   → (256128,)     sums to 1.0

prediction = argmax(probabilities)  → the token with highest probability
```

### The Unembedding Direction (crucial for everything that follows)

For the SVA task, we only care about two tokens: the correct verb (e.g., "is") and the incorrect verb (e.g., "are"). The **unembedding direction** is:

```
unembed_dir = W_U[:, token_id("is")] - W_U[:, token_id("are")]     → (2048,)
```

This is a single vector in the model's 2048-dimensional space. If the final residual stream vector **points along this direction**, the model predicts "is". If it points **against** this direction, the model predicts "are".

The **logit difference** is just the dot product:
```
logit_diff = final_residual @ unembed_dir
           = logit("is") - logit("are")
```

Positive = model prefers "is" (correct for singular subject).
Negative = model prefers "are" (correct for plural subject).

### Why this is so powerful

Because the final residual stream is a **sum** of all component outputs, the logit difference decomposes perfectly:

```
logit_diff = (embedding + head_0_0 + head_0_1 + ... + mlp_0 + head_1_0 + ... + mlp_17) @ unembed_dir
           = embedding @ unembed_dir     ← embedding's contribution
           + head_0_0 @ unembed_dir      ← L0H0's contribution  
           + head_0_1 @ unembed_dir      ← L0H1's contribution
           + ...
           + mlp_17 @ unembed_dir        ← MLP17's contribution
```

Each term is one component's **Direct Logit Attribution (DLA)** — and the sum is exact. No approximation. This is the foundation of Experiment 2.

## 1.7 TransformerLens Hooks — How We Intercept the Model

TransformerLens wraps the transformer so we can **read** or **modify** any internal activation during a forward pass. Every intermediate computation has a named "hook point."

The key hooks we use:

| Hook Name | Shape | What It Is |
|-----------|-------|------------|
| `blocks.{L}.attn.hook_z` | `(batch, seq, n_heads, d_head)` | Per-head output **before** W_O projection. This is the 256-dim vector each head produces. The main hook for patching, PCA, steering. |
| `blocks.{L}.hook_attn_out` | `(batch, seq, d_model)` | Sum of all heads' outputs after W_O projection. The attention layer's total contribution to the residual stream. |
| `blocks.{L}.hook_mlp_out` | `(batch, seq, d_model)` | MLP block's output — its total contribution to the residual stream. |
| `blocks.{L}.mlp.hook_post` | `(batch, seq, d_mlp)` | Post-activation neuron values inside the MLP (after gating). Used for per-neuron analysis. |
| `blocks.{L}.hook_resid_post` | `(batch, seq, d_model)` | The residual stream after layer L (includes everything up to this point). |

### Two ways to use hooks:

**1. Read (caching):** Run the model and save activations
```python
logits, cache = model.run_with_cache(tokens)
head_output = cache["blocks.13.attn.hook_z"]   # just read it
```

**2. Write (intervening):** Modify activations during the forward pass
```python
def my_hook(value, hook):
    value[:, -1, 7, :] += steering_vector   # modify head 7's output
    return value

logits = model.run_with_hooks(tokens, fwd_hooks=[
    ("blocks.13.attn.hook_z", my_hook)
])
```

This hooking mechanism is what makes every experiment possible — patching swaps activations, steering adds vectors, DLA reads outputs.

---

# Part 2: The Task — Subject-Verb Agreement (SVA)

## 2.1 What Is SVA?

Subject-verb agreement means the verb must match the subject's grammatical number:
- "The doctor **is** tall" (singular subject → singular verb)
- "The doctors **are** tall" (plural subject → plural verb)

This seems trivial, but it gets tricky with **relative clauses** that put a distractor noun between the subject and verb:

> "The doctor **that helped the teachers** ___"

The model must predict "**is**" (agreeing with "doctor", singular) and NOT "**are**" (which would agree with "teachers", the distractor). This requires the model to track which noun is the actual subject through intervening material.

## 2.2 Contrastive Pairs

Every example in our dataset is a **contrastive pair** — two sentences identical except for the subject's number:

```
clean:     "The doctor that helped the teacher is"      → good_verb = "is"
corrupted: "The doctors that helped the teacher are"    → bad_verb  = "are"  
```

Why pairs? They let us isolate the **minimal causal signal**. Everything is the same except one thing (singular vs plural subject), so any difference in the model's behavior **must** be caused by that one change. This is the experimental control.

## 2.3 The Logit Difference Metric

For each example, we measure:

```python
logit_diff = logit("is") - logit("are")
```

- **Positive** logit_diff → model correctly prefers the singular verb
- **Negative** logit_diff → model incorrectly prefers the plural verb

We always compute this on the **clean** sentence (singular subject), so a positive value means the model gets it right.

The code in `metrics.py`:
```python
def logit_diff(logits, good_token_ids, bad_token_ids, pos=-1):
    last_logits = logits[:, pos, :]                    # logits at final position
    good = last_logits.gather(1, good_token_ids)       # logit for correct verb
    bad = last_logits.gather(1, bad_token_ids)         # logit for incorrect verb
    return good - bad
```

---

# Part 3: The Experiments

## Overview

Each experiment answers a different question about how the model solves SVA. They build on each other:

```
Experiment 1: ACTIVATION PATCHING
    "Which components CAUSALLY CARRY the number signal?"
    → Answer: L13H7 and L17H4
         |
         v
Experiment 2: DIRECT LOGIT ATTRIBUTION (DLA)
    "Which components DIRECTLY PUSH toward the correct verb?"
    → Answer: L13H7 and L17H4 (confirms patching)
         |
         v
Experiment 3: NEURON ANALYSIS
    "Which specific MLP NEURONS contribute to the decision?"
    → Answer: Neuron 2069 in MLP13, Neuron 1138 in MLP17
         |
         v
Experiment 4: PCA
    "Does L13H7 encode number as a SINGLE DIRECTION?"
    → Answer: Yes, PC1 separates singular from plural
         |
         v
Experiment 5: CROSS-LINGUAL STEERING
    "Does the ENGLISH direction causally control OTHER LANGUAGES?"
    → Answer: Yes — proving the representation is language-independent
         |
         v
Experiment 6: EDGE ATTRIBUTION PATCHING
    "Can we find the circuit FASTER and include MLPs?"
    → Answer: Yes, and MLPs are actually more important than attention!
         |
         v
Experiment 7: WEIGHT-LEVEL ANALYSIS
    "What do the WEIGHT MATRICES tell us without running any data?"
    → Answer: The model's weights are structurally wired for SVA
         |
         v
Experiment 8: CROSS-LINGUAL GEOMETRY
    "WHERE in the network do languages CONVERGE?"
    → Answer: Layer 13 — exactly where the SVA circuit lives
```

---

## Experiment 1: Activation Patching

### The Question
Which attention heads **causally carry** the subject-number signal? 

"Causally carry" means: if we remove this head's contribution, does the model lose the ability to do SVA? This is stronger than correlation — it's a causal intervention.

### The Intuition

Imagine the model is a factory assembly line with 144 workers (heads). You want to find which workers are actually doing the important work. Your strategy:

1. Run the assembly line with a **defective input** (corrupted sentence — plural subject). The output is wrong.
2. For each worker, secretly replace their defective work with a copy of what they would have produced with a **good input** (clean sentence — singular subject).
3. If swapping worker X's output fixes the final product → Worker X was the one doing the critical work.

### The Math

```
For each example:
    1. Run model on CLEAN input → cache all activations
       clean_logit_diff = logit("is") - logit("are")    [positive — correct]
    
    2. Run model on CORRUPTED input
       corrupted_logit_diff = logit("is") - logit("are") [negative — wrong]
    
    3. For each head (layer L, head H):
       - Run model on CORRUPTED input BUT replace head L,H's activation
         with the CLEAN version
       - patched_logit_diff = new logit("is") - logit("are")
       
       - normalized_effect = (patched - corrupted) / (clean - corrupted)
         
         0.0 = patching had no effect (this head doesn't matter)
         1.0 = patching fully restored correct behavior (this head is critical)
```

### The Code (from `patching.py`)

The core patching loop. For each of 144 heads, we hook into `hook_z` and swap one head's activation:

```python
def _patch_head(model, corrupted_tokens, clean_cache, layer, head, good_ids, bad_ids):
    hook_name = f"blocks.{layer}.attn.hook_z"
    
    def hook_fn(value, hook):
        # value shape: (batch, seq_len, n_heads, d_head)
        # Only replace THIS head at the FINAL position
        # Leave all other heads untouched
        value[:, -1, head, :] = clean_cache[hook_name][:, -1, head, :]
        return value
    
    with torch.no_grad():
        patched_logits = model.run_with_hooks(
            corrupted_tokens,
            fwd_hooks=[(hook_name, hook_fn)]
        )
    return logit_diff(patched_logits, good_ids, bad_ids).item()
```

**What happens physically:**
1. The model starts processing the corrupted sentence ("The doctors that helped the teacher")
2. Every head computes normally on the corrupted input
3. EXCEPT: when we reach layer L head H, our hook fires and replaces `value[:, -1, head, :]` with the clean version
4. From this point on, the model processes with head L,H having the "right" information
5. We measure: did the output improve?

### Why we only patch the final position (`-1`)

The model predicts the next token at the **last** position. That's where the verb logits are computed. The number signal needs to reach this position to influence the prediction. So we patch at position -1 to ask: "does this head carry number information to the prediction site?"

In [ ]:
# Let's look at the actual patching results
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
os.chdir("/home/vacl2/similarity")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, lang, title in zip(axes, ["en", "es"], ["English", "Spanish"]):
    d = np.load(f"results/patching_{lang}.npz")
    head_out = d["head_out"]  # shape: (18 layers, 8 heads)
    
    sns.heatmap(head_out, ax=ax, cmap="RdYlBu_r", center=0,
                xticklabels=[f"H{h}" for h in range(8)],
                yticklabels=[f"L{l}" for l in range(18)],
                vmin=-0.1, vmax=0.6)
    ax.set_title(f"Activation Patching — {title}\n(normalized patch effect)", fontsize=12)
    ax.set_xlabel("Head")
    ax.set_ylabel("Layer")
    
    # Annotate top heads
    top_idx = np.unravel_index(np.argsort(head_out.flatten())[-3:], head_out.shape)
    for l, h in zip(*top_idx):
        ax.text(h + 0.5, l + 0.5, f"{head_out[l,h]:.2f}", ha="center", va="center",
                fontsize=8, fontweight="bold", color="white")

plt.tight_layout()
plt.suptitle("L13H7 dominates in BOTH languages — the circuit is shared", y=1.02, fontsize=13)
plt.show()

print("English top heads:")
for l, h in zip(*np.unravel_index(np.argsort(head_out.flatten())[-5:][::-1], head_out.shape)):
    print(f"  L{l}H{h}: {head_out[l,h]:.3f}")

## Experiment 2: Direct Logit Attribution (DLA)

### The Question
Which components **directly push** the model's output toward the correct verb?

### How it differs from patching
- **Patching** is **causal**: it physically changes the forward pass and measures the effect. It tells you what *carries* the signal.
- **DLA** is **correlational**: it reads off each component's output and measures its alignment with the task direction. It tells you what *expresses* the signal in the output.

A head can carry a signal (high patching) without directly expressing it (low DLA) — for example, if it writes to the residual stream in a way that another head later reads and converts to the final output. And a head can have high DLA without high patching if its contribution is redundant (other heads write the same thing).

### The Math

Remember from Section 1.6: the logit difference decomposes as a sum over components.

For an attention head, the output lives in d_head space (256 dims) and needs to be projected to d_model space (2048 dims) via W_O before we can dot with the unembed direction:

```
For attention head at layer L, head H:
    z = hook_z[0, -1, H, :]              → (256,)   the head's raw output
    projected = z @ W_O[H]               → (2048,)  projected to model space
    head_DLA = projected @ unembed_dir   → scalar   contribution to logit diff
```

Note: in the code, this is written as `W_O[H].T @ z` which is mathematically identical for 1D vectors (`z @ W` == `W.T @ z`).

For MLP layers, the output is already in d_model space:
```
    mlp_out = hook_mlp_out[0, -1, :]     → (2048,)
    mlp_DLA = mlp_out @ unembed_dir      → scalar
```

### The Code (from `dla.py`)

```python
W_U = model.W_U  # (d_model, vocab_size)

for ex in examples:
    tokens, good_id, bad_id = tokenize_pair(model, ex["clean"], ...)
    _, cache = model.run_with_cache(tokens)
    
    # The direction that separates "is" from "are" in vocabulary space
    unembed_dir = W_U[:, good_id] - W_U[:, bad_id]   # (2048,)
    
    for layer in range(18):
        head_out = cache[f"blocks.{layer}.attn.hook_z"]  # (1, seq, 8, 256)
        W_O = model.blocks[layer].attn.W_O               # (8, 256, 2048)
        
        for head in range(8):
            z = head_out[0, -1, head, :]       # (256,) — this head's output
            projected = W_O[head].T @ z         # (2048,) — projected to model space
            dla = (projected @ unembed_dir)     # scalar — dot with task direction
            head_sums[layer, head] += dla
```

### Key insight: DLA tells you WHO is pushing, patching tells you WHO matters

A head with **high DLA but low patching** (like L0H5) outputs a large vector aligned with the verb direction, but its contribution is *redundant* — removing it doesn't change the prediction because other components compensate. 

A head with **high patching AND high DLA** (like L13H7) is both carrying the signal and directly expressing it — it's the core of the circuit.

In [ ]:
# DLA results — compare patching vs DLA side by side
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for col, lang, title in [(0, "en", "English"), (1, "es", "Spanish")]:
    # Patching
    pat = np.load(f"results/patching_{lang}.npz")["head_out"]
    sns.heatmap(pat, ax=axes[0, col], cmap="RdYlBu_r", center=0, vmin=-0.1, vmax=0.6,
                xticklabels=[f"H{h}" for h in range(8)],
                yticklabels=[f"L{l}" for l in range(18)])
    axes[0, col].set_title(f"Patching — {title}\n(causal importance)")
    
    # DLA
    dla = np.load(f"results/dla_{lang}.npz")["head_dla"]
    sns.heatmap(dla, ax=axes[1, col], cmap="RdBu_r", center=0,
                xticklabels=[f"H{h}" for h in range(8)],
                yticklabels=[f"L{l}" for l in range(18)])
    axes[1, col].set_title(f"DLA — {title}\n(direct logit contribution)")

plt.tight_layout()
plt.suptitle("Patching (causal) vs DLA (correlational)\n"
             "L0H5 has huge DLA but zero patching — its contribution is redundant", 
             y=1.04, fontsize=13)
plt.show()

# Print the comparison for key heads
print("\nKey heads — Patching vs DLA:")
print(f"{'Head':<8} {'EN Patch':>10} {'EN DLA':>10} {'ES Patch':>10} {'ES DLA':>10}")
print("-" * 50)
en_pat = np.load("results/patching_en.npz")["head_out"]
en_dla = np.load("results/dla_en.npz")["head_dla"]
es_pat = np.load("results/patching_es.npz")["head_out"]
es_dla = np.load("results/dla_es.npz")["head_dla"]
for (l, h) in [(0, 5), (13, 7), (17, 4)]:
    print(f"L{l}H{h:<5} {en_pat[l,h]:>10.3f} {en_dla[l,h]:>10.3f} {es_pat[l,h]:>10.3f} {es_dla[l,h]:>10.3f}")

## Experiment 3: Neuron Analysis

### The Question
Within the important MLP layers (13 and 17), which **individual neurons** contribute most to the SVA decision?

### Why this matters
Patching and DLA told us MLP13 and MLP17 are important at the layer level. But each MLP has **16,384 neurons**. Which specific ones are doing the work? If it's just a handful, that's evidence of a clean, sparse circuit. If it's thousands, the computation is distributed and harder to interpret.

### The Math

This is DLA decomposed one level deeper. Instead of asking "what does MLP13 contribute?", we ask "what does neuron 2069 in MLP13 contribute?"

Each neuron's contribution is the product of two things:
```
neuron_DLA[i] = post[i]  *  (W_out[i, :] @ unembed_dir)
                ^^^^^^^^     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                activation:   alignment:
                how strongly  how much this neuron's output
                this neuron   column points toward the
                fired         verb-number direction
```

- `post[i]` is the neuron's activation value after gating (from `hook_post`)
- `W_out[i, :]` is the neuron's output row — the direction it writes to the residual stream
- `unembed_dir` is the verb-number direction

If both are large and point the same way, this neuron is pushing toward the correct verb.

### The Code (from `neurons.py`)

```python
for ex in examples:
    tokens, good_id, bad_id = tokenize_pair(model, ex["clean"], ...)
    unembed_dir = (W_U[:, good_id] - W_U[:, bad_id]).float()
    
    _, cache = model.run_with_cache(tokens, 
                                     names_filter=["blocks.13.mlp.hook_post"])
    
    for layer in [13, 17]:
        post = cache[f"blocks.{layer}.mlp.hook_post"][0, -1, :]  # (16384,)
        W_out = model.blocks[layer].mlp.W_out                     # (16384, 2048)
        
        # How aligned is each neuron's output with the task direction?
        w_proj = W_out @ unembed_dir                               # (16384,)
        
        # Neuron DLA = activation * alignment (element-wise)
        neuron_dla = (post * w_proj).detach().cpu().numpy()        # (16384,)
        neuron_sums[layer] += neuron_dla
```

### What does this tell us?

If neuron 2069 has large positive DLA, it means:
1. It **fires strongly** when the subject is singular (high `post[2069]`)
2. Its **output column** in W_out points toward the "singular verb" direction
3. Together, it's actively pushing the model to predict "is" over "are"

This is how we identify the specific neurons that implement the number agreement computation inside the MLP.

In [ ]:
# Neuron analysis results
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

en_neurons = np.load("results/neurons_en.npz")
for ax, layer_key, title in zip(axes, ["layer_13", "layer_17"], ["MLP Layer 13", "MLP Layer 17"]):
    scores = en_neurons[layer_key]  # (16384,) — one score per neuron
    
    # Sort by absolute DLA
    sorted_idx = np.argsort(np.abs(scores))[::-1]
    top_50 = sorted_idx[:50]
    
    colors = ['#e74c3c' if scores[i] > 0 else '#3498db' for i in top_50]
    ax.bar(range(50), [scores[i] for i in top_50], color=colors, width=0.8)
    ax.set_xlabel("Neuron rank (by |DLA|)")
    ax.set_ylabel("Neuron DLA")
    ax.set_title(f"{title} — Top 50 neurons (EN)\nRed=pushes correct, Blue=pushes incorrect")
    ax.axhline(y=0, color='gray', linewidth=0.5)
    
    # Label the top neuron
    top_neuron = sorted_idx[0]
    ax.annotate(f"Neuron {top_neuron}\nDLA={scores[top_neuron]:.3f}", 
                xy=(0, scores[top_neuron]), fontsize=9,
                xytext=(10, scores[top_neuron]*0.7))

plt.tight_layout()
plt.show()

# Print top neurons
print("MLP Layer 13 — Top 10 neurons:")
for rank, idx in enumerate(sorted_idx[:10]):
    print(f"  #{rank+1:2d}  neuron {idx:5d}  DLA={scores[idx]:+.4f}")

## Experiment 4: PCA on L13H7 Outputs

### The Question
Does L13H7 encode subject number as a **single direction** in its 256-dimensional output space?

### The Intuition

L13H7 outputs a 256-dimensional vector at the final position. We have two types of inputs — singular subjects and plural subjects. If the head encodes number as a single direction, then:
- All singular-subject vectors should cluster on one side of that direction
- All plural-subject vectors should cluster on the other side
- PCA should find this as PC1 (the direction of maximum variance)

Think of it like this: imagine 256-dimensional space. If singular and plural vectors form two clouds separated by a hyperplane, PCA finds the axis perpendicular to that hyperplane.

```
    PC1 axis
    ════════════════════════════════►
    
    ○ ○ ○ ○ ○ ○           ● ● ● ● ● ●
    ○ ○ ○ ○ ○             ● ● ● ● ●
    singular cluster       plural cluster
```

### The Math

```
1. For each example, run both the singular and plural versions
2. Extract L13H7's output: cache["blocks.13.attn.hook_z"][0, -1, 7, :]  → (256,)
3. Collect all vectors with labels (0=singular, 1=plural)
4. Fit PCA on the vectors
5. Project onto PC1 — does it separate singular from plural?
```

PCA finds the directions of maximum variance. If singular/plural is the biggest source of variation in the data, PC1 will be the number direction.

### The Code (from `pca.py`)

```python
def collect_head_outputs(model, examples, layer, head, device):
    hook_name = f"blocks.{layer}.attn.hook_z"
    vectors, labels, langs = [], [], []
    
    for ex in examples:
        for use_corrupted, label in [(False, 0), (True, 1)]:
            # label 0 = clean (singular subject)
            # label 1 = corrupted (plural subject)
            prompt = ex["corrupted"] if use_corrupted else ex["clean"]
            tokens, _, _ = tokenize_pair(model, prompt, ...)
            
            _, cache = model.run_with_cache(tokens, names_filter=[hook_name])
            h_out = cache[hook_name][0, -1, head, :]  # (256,) — the head's output
            
            vectors.append(h_out.numpy())
            labels.append(label)        # 0 or 1
            langs.append(ex["lang"])    # "en", "es", etc.
    
    return np.stack(vectors), np.array(labels), np.array(langs)

# Fit PCA
pca = PCA(n_components=10)
pca.fit(vectors)                          # find the directions of max variance
projections = pca.transform(vectors)[:, 0] # project onto PC1
pc1 = pca.components_[0]                   # the PC1 direction vector (256,)
```

### The Earlier Bug (Now Fixed)

An earlier run used `--lang all` (all 7 languages), so PC1 captured **language identity** (the biggest source of variance across 7 languages) instead of **number**. The fix was straightforward: run with `--lang en --max-examples 50` (matching the paper). The corrected PCA produces a clean separation of 3.29 between singular and plural clusters.

### Why this matters for the story

If PCA on English data alone finds a direction that separates singular from plural, AND that same direction also separates singular from plural in Spanish/French/Russian/etc., then the model uses a **language-independent** representation of grammatical number. That's the core claim of the paper.

In [ ]:
# PCA results -- correctly fitted on English only (50 examples, matching paper)
d = np.load("results/pca_L13H7.npz")
proj = d["projections"]
labels = d["labels"]
langs = d["langs"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: histogram showing clean separation
ax = axes[0]
sg = proj[labels == 0]
pl = proj[labels == 1]
ax.hist(sg, bins=20, alpha=0.6, color="#e74c3c", label=f"Singular (mean={sg.mean():.2f})")
ax.hist(pl, bins=20, alpha=0.6, color="#3498db", label=f"Plural (mean={pl.mean():.2f})")
ax.set_xlabel("PC1 projection")
ax.set_ylabel("Count")
ax.set_title(f"PC1 cleanly separates singular from plural\nSeparation = {abs(sg.mean()-pl.mean()):.2f}")
ax.legend()
ax.axvline(x=0, color="gray", linestyle="--", alpha=0.5)

# Right: compare old (all-language) vs new (EN-only)
ax = axes[1]
try:
    d_old = np.load("results/pca_L13H7_old_all_langs.npz")
    old_sg = d_old["projections"][d_old["labels"] == 0]
    old_pl = d_old["projections"][d_old["labels"] == 1]
    old_sep = abs(old_sg.mean() - old_pl.mean())
    new_sep = abs(sg.mean() - pl.mean())
    ax.bar(["Old (all 7 langs)", "New (EN only, 50 ex)"], [old_sep, new_sep],
           color=["#e74c3c", "#2ecc71"])
    ax.set_ylabel("Singular/Plural separation on PC1")
    ax.set_title(f"Fix: {old_sep:.3f} -> {new_sep:.2f} ({new_sep/old_sep:.0f}x improvement)")
    for i, v in enumerate([old_sep, new_sep]):
        ax.text(i, v + 0.05, f"{v:.3f}", ha="center", fontsize=11, fontweight="bold")
except FileNotFoundError:
    ax.text(0.5, 0.5, "Old PCA backup not found", ha="center", va="center", transform=ax.transAxes)

plt.suptitle("PCA on L13H7 -- English only, 50 examples (matching paper)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"PC1 separation: {abs(sg.mean()-pl.mean()):.3f}")
print(f"Singular: mean={sg.mean():.3f}, std={sg.std():.3f}")
print(f"Plural:   mean={pl.mean():.3f}, std={pl.std():.3f}")

## Experiment 5: Cross-Lingual Steering

### The Question
Does the English-derived number direction **causally control** verb predictions in other languages?

### The Intuition

This is the climax of the paper's argument. We've found a direction (PC1 from English PCA) that separates singular from plural in L13H7's output space. Now we ask: if we **inject** this direction into the model while it processes a Spanish sentence, does it flip the verb from "es" to "son" (or vice versa)?

If yes → the model uses the **same internal representation** for grammatical number in both languages. The representation is language-independent.

### The Math

```
For a Spanish sentence "El doctor que habló al maestro":
    1. Run model normally → model predicts "es" (singular) ✓
    
    2. Add α × PC1 to L13H7's output at the final position:
       hook_z[:, -1, 7, :] += α * pc1_vector
       
    3. Run model with this intervention → does it now predict "son" (plural)?
    
    4. Sweep α from 0 to 50, measure flip rate at each magnitude
```

The **flip rate** is the fraction of examples where the prediction changes due to steering. Higher α = stronger push along the number direction = more flips.

### The Code (from `steering.py`)

```python
def steer_and_measure(model, examples, pc1, layer, head, alpha, direction, device):
    hook_name = f"blocks.{layer}.attn.hook_z"
    pc1_tensor = torch.tensor(pc1, dtype=torch.float32, device=device)
    sign = +1.0 if direction == "pos" else -1.0
    
    flips = 0
    total = 0
    
    for ex in examples:
        tokens, good_id, bad_id = tokenize_pair(model, ex["clean"], ...)
        
        # Baseline: what does the model predict without intervention?
        base_logits = model(tokens)
        base_pred = 0 if base_logits[0,-1,good_id] > base_logits[0,-1,bad_id] else 1
        
        # Steered: add α * PC1 to L13H7's output
        def hook_fn(value, hook):
            value[:, -1, head, :] += sign * alpha * pc1_tensor
            return value
        
        steered_logits = model.run_with_hooks(tokens, 
                                               fwd_hooks=[(hook_name, hook_fn)])
        steered_pred = 0 if steered_logits[0,-1,good_id] > steered_logits[0,-1,bad_id] else 1
        
        if steered_pred != base_pred:
            flips += 1
        total += 1
    
    return flips / total
```

### Our results

With the corrected English-only PC1 direction:
- At α=0: 0% flip rate (no intervention)
- At α=5-10: 34-66% flip rate (already strong at low doses!)
- At α=50: 20-64% depending on language

Swahili shows the strongest effect (64.2% at α=50), followed by French (45.0%) and Russian (40.0%). The direction transfers across all 6 target languages, from 5 different language families, including Cyrillic script (Russian) and prefix-based agreement (Swahili).

In [ ]:
# Steering results -- with corrected English-only PCA direction
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
lang_names = {"es": "Spanish", "fr": "French", "ru": "Russian", 
              "tr": "Turkish", "sw": "Swahili", "qu": "Quechua"}

for ax, lang in zip(axes.flatten(), ["es", "fr", "ru", "tr", "sw", "qu"]):
    d = np.load(f"results/steering_{lang}.npz")
    alphas = d["alphas"]
    ax.plot(alphas, d["flip_rate_pos"], "o-", color="#e74c3c", label="positive (+a)")
    ax.plot(alphas, d["flip_rate_neg"], "s-", color="#3498db", label="negative (-a)")
    ax.set_xlabel("Steering magnitude a")
    ax.set_ylabel("Flip rate")
    ax.set_title(f"Steering -> {lang_names[lang]}")
    ax.legend(fontsize=8)
    ax.set_ylim(-0.02, 0.75)
    ax.grid(alpha=0.3)

plt.suptitle("Cross-lingual steering with corrected English-only PC1 direction\n"
             "Strong causal transfer: Swahili 64%, French 45%, Spanish 47% at moderate a",
             fontsize=13, y=1.03)
plt.tight_layout()
plt.show()

# Summary table
print("Max flip rate at a=50 (new corrected results):")
for lang in ["es", "fr", "ru", "tr", "sw", "qu"]:
    d = np.load(f"results/steering_{lang}.npz")
    max_flip = max(d["flip_rate_pos"].max(), d["flip_rate_neg"].max())
    print(f"  {lang_names[lang]:>10}: {max_flip:.1%}")

## Experiment 6: Edge Attribution Patching (EAP)

### The Question
Can we find the important components **faster** and also include **MLPs** in the analysis?

### The Problem with Standard Patching

Standard activation patching (Experiment 1) requires a separate forward pass for every component we want to test:
- 18 layers × 8 heads = 144 forward passes per example (just for heads)
- × 128 examples = 18,432 forward passes total
- And that's only attention heads — adding 18 MLP blocks would be even more

EAP uses calculus to estimate the same thing with just **3 passes per example** (2 forward + 1 backward).

### The Math

The insight: the effect of patching component C is approximately:

```
patch_effect(C) ≈ Δactivation(C) × ∂(logit_diff)/∂(activation(C))
                  ^^^^^^^^^^^^^^^^   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                  how different is    how sensitive is the output
                  clean vs corrupted  to changes in this component
                  at this component
```

This is a **first-order Taylor approximation**. It says: "if I changed this activation by a small amount Δ, the output would change by approximately Δ × gradient."

- `Δactivation = clean_activation - corrupted_activation` (how much the signal differs)
- `gradient = ∂(logit_diff)/∂(activation)` (how much the output cares about this activation)

### The Code (from `edge_patching.py`)

```python
for ex in examples:
    # Step 1: Clean forward pass (no gradients) — get reference activations
    with torch.no_grad():
        _, clean_cache = model.run_with_cache(clean_tokens)
    
    # Step 2: Corrupted forward pass WITH gradients — track everything
    activations = {}
    def make_fwd_hook(name):
        def hook_fn(value, hook):
            activations[name] = value
            value.requires_grad_(True)    # enable gradient tracking!
            value.retain_grad()           # keep the gradient after backward
            return value
        return hook_fn
    
    # Hook every component
    hook_pairs = []
    for layer in range(18):
        hook_pairs.append(("blocks.{layer}.attn.hook_z", make_fwd_hook(f"head_{layer}")))
        hook_pairs.append(("blocks.{layer}.hook_mlp_out", make_fwd_hook(f"mlp_{layer}")))
    
    corrupted_logits = model.run_with_hooks(corrupted_tokens, fwd_hooks=hook_pairs)
    
    # Step 3: Backward pass — compute gradients
    ld = logit_diff(corrupted_logits, good_ids, bad_ids)
    ld.backward()   # now every hooked activation has a .grad attribute
    
    # Step 4: Score = |Δactivation × gradient|
    for layer in range(18):
        for head in range(8):
            delta = clean_z[0,-1,head,:] - corrupted_z[0,-1,head,:]  # how much it changed
            grad = corrupted_z.grad[0,-1,head,:]                      # how much output cares
            score = (delta * grad).abs().sum()                         # element-wise product, summed
```

### Why EAP reveals something patching missed

Standard patching only tested **attention heads** (144 components). EAP tests **attention heads AND MLPs** (144 + 18 = 162 components) in the same analysis. The result was surprising: **MLP layers dominate**. MLP17 has the highest EAP score in every single language — higher than any attention head. This means the MLPs are doing critical SVA computation that was invisible to the attention-only patching analysis.

In [ ]:
# EAP results — attention heads vs MLPs
fig, ax = plt.subplots(figsize=(14, 6))

langs = ["en", "es", "fr", "ru", "tr", "sw", "qu"]
lang_names_full = {"en": "English", "es": "Spanish", "fr": "French", "ru": "Russian", 
                   "tr": "Turkish", "sw": "Swahili", "qu": "Quechua"}
x_offset = np.arange(len(langs))
width = 0.12

# For each language, show top 5 components
for i, lang in enumerate(langs):
    d = np.load(f"results/edge_patching_{lang}.npz")
    labels = [str(x) for x in d["component_labels"]]
    scores = d["node_scores"]
    
    # Normalize scores to max=1 for comparison
    scores_norm = scores / scores.max()
    
    # Separate heads and MLPs
    head_max = max(scores_norm[j] for j, l in enumerate(labels) if "H" in l)
    mlp_max = max(scores_norm[j] for j, l in enumerate(labels) if l.startswith("MLP"))
    
    # Plot top MLP and top head
    top_mlp_idx = max((j for j, l in enumerate(labels) if l.startswith("MLP")), key=lambda j: scores_norm[j])
    top_head_idx = max((j for j, l in enumerate(labels) if "H" in l), key=lambda j: scores_norm[j])
    
    ax.bar(i - 0.15, scores_norm[top_mlp_idx], width=0.25, color="#4ecdc4", 
           label="Top MLP" if i == 0 else None)
    ax.bar(i + 0.15, scores_norm[top_head_idx], width=0.25, color="#ff6b6b",
           label="Top Attn Head" if i == 0 else None)
    
    ax.text(i - 0.15, scores_norm[top_mlp_idx] + 0.02, labels[top_mlp_idx], 
            ha="center", fontsize=8, rotation=45)
    ax.text(i + 0.15, scores_norm[top_head_idx] + 0.02, labels[top_head_idx], 
            ha="center", fontsize=8, rotation=45)

ax.set_xticks(range(len(langs)))
ax.set_xticklabels([lang_names_full[l] for l in langs])
ax.set_ylabel("Normalized EAP score")
ax.set_title("Edge Attribution Patching: MLPs dominate in EVERY language\n"
             "MLP17 is always #1 — invisible to attention-only patching")
ax.legend()
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## Experiment 7: Weight-Level Circuit Analysis

### The Question
What can the **weight matrices alone** tell us about the SVA circuit — without running any data?

### The Intuition

All the experiments so far required running data through the model. But the weights are fixed — they don't change. Can we look at the weights directly and predict which heads will be important for SVA?

The answer is yes, using the OV circuit from Section 1.4.

### Part A: Head Importance from OV + Task Projection

Each head has an OV matrix = W_V @ W_O, shape (d_model, d_model). We project it onto the unembed direction:

```
task_weight = OV @ unembed_dir      → (d_model,)

head_importance = ||task_weight||    → scalar
```

This measures: "how strongly is this head's OV circuit aligned with the verb-number direction?" A head with high importance has weights that are structurally oriented to process number agreement.

### Part B: Per-Weight Importance

We can go deeper and score **every individual weight** in W_V and W_O:

For W_O (shape d_head × d_model):
```
wo_importance[j, k] = |W_O[j, k]| × |unembed_dir[k]|
```
Weight W_O[j,k] matters if it's large AND the unembed direction has a large component in dimension k.

For W_V (shape d_model × d_head):
```
downstream[j] = Σ_k |W_O[j, k] × unembed_dir[k]|    ← how important is d_head dim j?
wv_importance[i, j] = |W_V[i, j]| × downstream[j]
```
Weight W_V[i,j] matters if it's large AND the d_head dimension j it feeds into is important for the task.

### Part C: Cross-Layer Connections

The big question for the flow visualization: how do heads **wire to each other**?

```
For head A (layer L) and head B (layer L'):
    OV_A = W_V_A @ W_O_A
    OV_B = W_V_B @ W_O_B
    
    output_dir_A = OV_A @ unembed_dir      "what A writes that matters for SVA"
    input_dir_B  = OV_B^T @ unembed_dir    "what B reads that matters for SVA"
    
    connection(A→B) = |cos(output_dir_A, input_dir_B)|
```

If A's task-relevant output aligns with B's task-relevant input, A feeds into B for this task. This is purely a property of the weights — no data needed.

### The Code (from `circuit_map.py`)

```python
def compute_cross_layer_connections(model, unembed_dir):
    output_dirs = []
    input_dirs = []
    
    for layer in range(18):
        for head in range(8):
            W_V = model.W_V[layer, head]    # (d_model, d_head)
            W_O = model.W_O[layer, head]    # (d_head, d_model)
            ov = W_V @ W_O                  # (d_model, d_model)
            
            out_dir = ov @ unembed_dir      # (d_model,) — what this head writes
            in_dir = ov.T @ unembed_dir     # (d_model,) — what this head reads
            
            output_dirs.append(out_dir)
            input_dirs.append(in_dir)
    
    # Connection strength = cosine similarity of output_A and input_B
    for i in range(144):
        for j in range(144):
            if layer_of[i] < layer_of[j]:   # only forward connections
                connection[i,j] = |cos(output_dirs[i], input_dirs[j])|
```

### The Key Finding

English routes the SVA signal through **L13→L17** (L13H3→L17H4 = 0.41, L13H7→L17H4 = 0.37).

All other languages route through **L14→L16** (L14H0→L16H1, L14H3→L16H0).

This is entirely a property of the **unembed direction** — the only language-specific input. The English verb pair ("is"/"are") has an unembed direction that aligns with the L13→L17 pathway, while all other verb pairs align with L14→L16. The weights are the same; the difference is which pathway each language's verb tokens "light up."

In [ ]:
# Weight importance and connection maps
import re

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Head importance heatmaps for EN vs ES
for ax, lang, title in zip(axes, ["en", "es"], ["English", "Spanish"]):
    d = np.load(f"results/circuit_map_{lang}.npz")
    hi = d["head_importance"]
    sns.heatmap(hi, ax=ax, cmap="YlOrRd", 
                xticklabels=[f"H{h}" for h in range(8)],
                yticklabels=[f"L{l}" for l in range(18)])
    ax.set_title(f"Weight Importance — {title}\n(||OV @ unembed_dir||)")

plt.suptitle("Weight-level head importance (no data needed — pure weight analysis)", 
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Connection analysis
print("Top connections by language:")
print(f"{'Language':<10} {'Top Connection':<25} {'Strength':>10}")
print("-" * 48)
for lang in ["en", "es", "fr", "ru"]:
    d = np.load(f"results/circuit_map_{lang}.npz")
    conn = d["connection_matrix"]
    labels = [str(x) for x in d["connection_labels"]]
    
    best_val, best_src, best_dst = 0, "", ""
    for i in range(len(labels)):
        for j in range(len(labels)):
            ms = re.match(r"L(\d+)H(\d+)", labels[i])
            md = re.match(r"L(\d+)H(\d+)", labels[j])
            if ms and md and int(ms[1]) < int(md[1]) and conn[i,j] > best_val:
                best_val = conn[i,j]
                best_src, best_dst = labels[i], labels[j]
    
    print(f"{lang:<10} {best_src} → {best_dst:<18} {best_val:>10.3f}")

## Experiment 8: Cross-Lingual Geometry (CKA)

### The Question
At which layer do the model's **representations converge** across languages?

### The Intuition

When the model processes "The doctor is" (English) and "El doctor es" (Spanish), the internal vectors at layer 0 are completely different — they start from different tokens. But by the time the model needs to predict the verb, it must have a representation of "singular subject, needs singular verb" regardless of language.

**CKA (Centered Kernel Alignment)** measures whether the **geometry** (the shape of the point cloud) of representations is similar between two languages at each layer.

### What CKA measures

Given N examples in English and N examples in Spanish, at layer L:
- English representations: X = (N × d_model) matrix
- Spanish representations: Y = (N × d_model) matrix

CKA doesn't compare individual vectors (they live in different token spaces). Instead, it compares the **pairwise similarity structure**:

```
K_X[i,j] = X[i] · X[j]    ← how similar are English examples i and j?
K_Y[i,j] = Y[i] · Y[j]    ← how similar are Spanish examples i and j?

CKA = HSIC(K_X, K_Y) / sqrt(HSIC(K_X, K_X) · HSIC(K_Y, K_Y))
```

where HSIC (Hilbert-Schmidt Independence Criterion) measures dependence between the two kernel matrices.

**CKA = 1.0**: the pairwise distances between examples are identical in both languages (same geometry)
**CKA = 0.0**: no structural similarity at all

### The Code (from `geometry.py`)

```python
def linear_cka(X, Y):
    """CKA between two representation matrices."""
    # Center the data
    X = X - X.mean(axis=0)
    Y = Y - Y.mean(axis=0)
    
    # Compute similarity matrices
    hsic_xy = np.linalg.norm(X.T @ Y, 'fro') ** 2
    hsic_xx = np.linalg.norm(X.T @ X, 'fro') ** 2
    hsic_yy = np.linalg.norm(Y.T @ Y, 'fro') ** 2
    
    return hsic_xy / np.sqrt(hsic_xx * hsic_yy)

# Collect residual stream at each layer for each language
def collect_layer_activations(model, examples, device):
    all_activations = np.zeros((n_layers, n_examples, d_model))
    
    for i, ex in enumerate(examples):
        _, cache = model.run_with_cache(tokens)
        for layer in range(18):
            all_activations[layer, i] = cache[f"blocks.{layer}.hook_resid_post"][0, -1, :]
    
    return all_activations

# Compute CKA between each language pair at each layer
for layer in range(18):
    for lang_a, lang_b in pairs:
        X = activations[lang_a][layer]   # (N, 2048)
        Y = activations[lang_b][layer]   # (N, 2048)
        cka[layer, pair_idx] = linear_cka(X, Y)
```

### The Result

CKA peaks at **Layer 13** (mean = 0.267 across all language pairs), exactly where the key SVA head lives. Languages converge at the critical processing layer, then diverge again as they head toward language-specific output tokens.

This is strong evidence that Layer 13 hosts a **shared, language-independent** representation of grammatical number.

In [ ]:
# CKA convergence — where do languages meet?
d = np.load("results/geometry.npz")
cka = d["cka_per_layer"]    # (18 layers, 21 pairs)
convergence = d["convergence"]  # (18,) — mean CKA per layer

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: convergence curve
ax = axes[0]
ax.plot(range(18), convergence, "o-", color="#8e44ad", linewidth=2, markersize=6)
ax.axvline(x=13, color="red", linestyle="--", alpha=0.5, label="Layer 13")
ax.fill_between(range(18), convergence, alpha=0.1, color="#8e44ad")
ax.set_xlabel("Layer")
ax.set_ylabel("Mean CKA (across all 21 language pairs)")
ax.set_title("Cross-lingual representation similarity\npeaks at Layer 13")
ax.legend()
ax.grid(alpha=0.3)

# Annotate peak
peak_layer = np.argmax(convergence)
ax.annotate(f"Peak: L{peak_layer}\nCKA={convergence[peak_layer]:.3f}",
            xy=(peak_layer, convergence[peak_layer]),
            xytext=(peak_layer + 2, convergence[peak_layer] + 0.01),
            arrowprops=dict(arrowstyle="->"), fontsize=10)

# Right: CKA heatmap (layer × pair)
ax = axes[1]
pair_labels = [str(x) for x in d["pair_labels"]]
# Show a subset of pairs for readability
subset = [0, 1, 2, 3, 4, 5]  # en-es, en-fr, en-ru, en-tr, en-sw, en-qu
sns.heatmap(cka[:, subset].T, ax=ax, cmap="viridis",
            xticklabels=[f"L{l}" for l in range(18)],
            yticklabels=[pair_labels[i] for i in subset])
ax.set_title("CKA by layer (English paired with each language)")
ax.set_xlabel("Layer")

plt.tight_layout()
plt.show()

## Experiment 9: Representation Engineering (RepE) — Where Does the Number Signal Emerge?

### The Question
At which layer does the model *first* encode subject number? Does the signal appear suddenly or build gradually? And critically: **does the signal emerge at the same relative depth in different models?**

### Why This Matters
All experiments so far tell us *which* components are important, but not *when* the signal forms. Understanding the layer-by-layer development of the number signal reveals the model's computational schedule — like watching a student solve a math problem step by step rather than just checking the final answer.

This experiment is inspired by **Representation Engineering** (Zou et al., 2023), which introduced the idea of extracting "reading vectors" — directions in representation space that encode specific concepts — using contrast pairs. Their key insight: instead of fitting PCA on raw activations (which captures variance from many sources), fit PCA on the **difference vectors** between contrastive pairs. This isolates the concept of interest (grammatical number) from everything else (word identity, position, language).

### The Method
For each layer in the model:
1. Run both the clean (singular) and corrupted (plural) sentence through the model
2. Extract the residual stream activation at the final position from both runs
3. Compute the **difference vector**: `clean_activation - corrupted_activation`
4. Collect these difference vectors across all examples
5. Run PCA on the differences — PC1 is the "reading vector" at that layer
6. Measure how well this reading vector classifies clean vs corrupted activations

This produces a **signal emergence profile**: a curve showing the strength of the number signal at each layer.

```
What we measure at each layer:
                                                    
  Accuracy   ─── How well the reading vector classifies clean vs corrupted
  Magnitude  ─── Mean absolute projection onto the reading vector  
  Exp. Var.  ─── How much of the difference variance is captured by PC1
  Diff Norm  ─── Total amount of change (clean vs corrupted) at this layer
```

### Why Difference Vectors (Not Raw Activations)?
Raw activations contain information about everything: the specific words, their positions, the language, the syntactic structure. PCA on raw activations might find "this is English" as PC1 rather than "this is singular." By taking differences between matched pairs, we cancel out everything shared between singular and plural sentences, leaving only the number signal.

```
Raw activations:  "The doctor ... is"   →  [word info + position + language + NUMBER + ...]
                  "The doctors ... are"  →  [word info + position + language + NUMBER + ...]
                                             ↓ subtract
Difference:                               [NUMBER signal only]
```

### Expected Results
The signal emergence profile for Gemma 2B should show:
- **Layers 0-5**: Low accuracy (~50%, chance level). The model hasn't started processing number yet.
- **Layers 6-10**: Gradually increasing. Number information begins to accumulate.
- **Layer 13**: Sharp peak. This is where L13H7 writes the number direction.
- **Layers 14-17**: Sustained or slightly declining. The signal persists for downstream use.

This profile is the model's "computational timeline" for SVA.

In [ ]:
# RepE signal emergence profiles
# These will be populated after running: uv run python -m circuits.repe --model gemma-2b --langs en es fr sw

import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
langs = ["en", "es", "fr", "sw"]
lang_names = {"en": "English", "es": "Spanish", "fr": "French", "sw": "Swahili"}
colors = {"en": "#2ecc71", "es": "#e74c3c", "fr": "#3498db", "sw": "#f39c12"}

for ax, lang in zip(axes.flat, langs):
    try:
        d = np.load(f"results/repe_gemma_2b_{lang}.npz")
        n_layers = len(d['accuracy'])
        layers = range(n_layers)
        
        ax.plot(layers, d['accuracy'], 'o-', color=colors[lang], linewidth=2, label='Accuracy')
        ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.3, label='Chance')
        ax.axvline(x=13, color='red', linestyle='--', alpha=0.3, label='Layer 13')
        ax.set_title(f'{lang_names[lang]} — Signal Emergence', fontsize=12, fontweight='bold')
        ax.set_xlabel('Layer')
        ax.set_ylabel('Classification Accuracy')
        ax.set_ylim(0.4, 1.05)
        ax.legend(fontsize=8)
    except FileNotFoundError:
        ax.text(0.5, 0.5, f'{lang_names[lang]}\n(pending)', ha='center', va='center', fontsize=14, transform=ax.transAxes)
        ax.set_title(f'{lang_names[lang]} — Signal Emergence', fontsize=12, fontweight='bold')

plt.suptitle('RepE Signal Emergence Profiles — Gemma 2B', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/figures/fig_repe_gemma.png', dpi=150, bbox_inches='tight')
plt.show()

## Experiment 10: Cross-Model Comparison — Gemma 2B vs BLOOM 3B

### The Question
Do different language models converge on the same computational strategy for multilingual SVA? Or is the circuit we found in Gemma an artifact of that specific architecture?

### Why BLOOM 3B?
BLOOM 3B is the ideal comparison model because it differs from Gemma 2B in almost every dimension:

| Property | Gemma 2B | BLOOM 3B |
|----------|----------|----------|
| Parameters | 2.5B | 3B |
| Layers | 18 | 30 |
| Heads/layer | 8 | 32 |
| d_model | 2048 | 2560 |
| d_head | 256 | 80 |
| Multilingual | English-dominant | 46 languages by design |
| Positional enc. | RoPE | ALiBi |
| Training data | Undisclosed | ROOTS corpus (documented) |

If both models route the number signal through the same relative depth and exhibit similar flow topology — despite all these architectural differences — that's strong evidence for a **universal computational principle**.

### The Comparison Framework
Since the two models have different architectures, we can't compare "layer 13 in Gemma" to "layer 13 in BLOOM." Instead, we normalize everything to **relative depth** — the fraction of the network traversed (0% = input, 100% = output).

#### Method 1: Signal Emergence Profile Comparison
Run RepE layer scanning on both models for the same languages. Normalize both profiles to relative depth [0, 1]. Compare the *shapes* of the curves.

```
Gemma (18 layers):  ░░░██████████████░░  peak at L13 (72%)
BLOOM (30 layers):  ░░░░░░█████████████████░░░  peak at L?? (?%)

Normalize to 0-100%:
Gemma:  ─────────╱██████████╲────  
BLOOM:  ─────────╱██████████╲────   ← same shape?
```

Metrics for comparing shapes:
- **Pearson correlation**: Do the curves rise and fall together? (r > 0.7 = similar)
- **Peak shift**: Do they peak at the same relative depth? (< 0.1 = aligned)
- **Cosine similarity**: Overall shape match? (> 0.8 = similar)

#### Method 2: Cross-Model CKA Heatmap
Feed the same sentences through both models. At each layer pair (Gemma layer i, BLOOM layer j), compute CKA between the two models' residual stream activations. This produces an alignment matrix.

```
          BLOOM layers →
       0  5  10  15  20  25  30
G  0   ■  .   .   .   .   .   .
e  3   .  ■   .   .   .   .   .
m  6   .  .   ■   .   .   .   .    ← A clear diagonal = aligned
m  9   .  .   .   ■   .   .   .       processing stages
a  12  .  .   .   .   ■   .   .
   15  .  .   .   .   .   ■   .
   18  .  .   .   .   .   .   ■
```

If the heatmap shows a clear diagonal band, the models process information in the same order. If it's scattered, they use fundamentally different strategies.

### Connection to the Platonic Representation Hypothesis
Huh et al. (2024) proposed that different neural networks trained on different data converge toward the same representation of reality — a "platonic" representation. Our experiment tests this for **internal computations**, not just final outputs. If the signal emergence profiles match, it suggests the computational *process* is also converging, not just the final representations.

### Datasets for Cross-Model Comparison
Not all languages work equally well with both models. After tokenizer filtering:

| Language | Gemma 2B | BLOOM 3B | Usable? |
|----------|----------|----------|---------|
| English | 536 | 536 | Yes |
| Spanish | 5,684 | 28,080 | Yes |
| French | 2,496 | 3,120 | Yes |
| Russian | 260 | 0 | No (BLOOM) |
| Turkish | 18 | 2 | No (too few) |
| Swahili | 2,366 | 2,366 | Yes |
| Quechua | 60 | 0 | No (BLOOM) |

The cross-model comparison uses **EN, ES, FR, SW** — four languages from three families (Germanic, Romance, Bantu), providing good typological diversity.

In [ ]:
# Cross-model comparison: Gemma 2B vs BLOOM 3B
# These will be populated after running: sbatch run_cross_model.sh

import numpy as np
import matplotlib.pyplot as plt

# --- Panel 1: Signal emergence profile overlay ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
langs = ["en", "es", "fr", "sw"]
lang_names = {"en": "English", "es": "Spanish", "fr": "French", "sw": "Swahili"}

for ax, lang in zip(axes.flat, langs):
    has_data = False
    for model, color, label, n_layers in [
        ("gemma_2b", "#2ecc71", "Gemma 2B", 18),
        ("bloom_3b", "#9b59b6", "BLOOM 3B", 30),
    ]:
        try:
            d = np.load(f"results/repe_{model}_{lang}.npz")
            acc = d['accuracy']
            rel_depth = np.linspace(0, 1, len(acc))
            ax.plot(rel_depth, acc, 'o-', color=color, linewidth=2,
                    label=f'{label} ({len(acc)}L)', markersize=4)
            has_data = True
        except FileNotFoundError:
            pass
    
    if not has_data:
        ax.text(0.5, 0.5, f'{lang_names[lang]}\n(pending)', ha='center',
                va='center', fontsize=14, transform=ax.transAxes)
    
    ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.3)
    ax.set_title(f'{lang_names[lang]}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Relative Depth (0=input, 1=output)')
    ax.set_ylabel('Reading Vector Accuracy')
    ax.set_ylim(0.4, 1.05)
    ax.legend(fontsize=8)

plt.suptitle('Signal Emergence: Gemma 2B vs BLOOM 3B (normalized depth)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/figures/fig_cross_model_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Panel 2: Cross-model CKA heatmap (if available) ---
try:
    cka_data = np.load('results/cross_cka_gemma-2b_bloom-3b_en.npz')
    cka_matrix = cka_data['cka_matrix']
    
    fig, ax = plt.subplots(figsize=(10, 7))
    im = ax.imshow(cka_matrix, cmap='magma', aspect='auto', origin='lower')
    ax.set_xlabel('BLOOM 3B Layer', fontsize=12)
    ax.set_ylabel('Gemma 2B Layer', fontsize=12)
    ax.set_title('Cross-Model CKA: Gemma 2B vs BLOOM 3B (English)',
                 fontsize=13, fontweight='bold')
    plt.colorbar(im, label='CKA Similarity')
    plt.tight_layout()
    plt.savefig('results/figures/fig_cross_model_cka.png', dpi=150, bbox_inches='tight')
    plt.show()
except FileNotFoundError:
    print('Cross-model CKA results not yet available. Run: sbatch run_cross_model.sh')

### Understanding the Results: What Does "Shape Matching" Mean?

When we say two models have "the same signal shape," here's what we mean concretely.

#### Signal Magnitude

At each layer, we measure: **how much does the model's internal state change between a singular and plural input?** This is the "signal magnitude" — the size of the difference vector at each layer.

Think of it like measuring the volume of a conversation about grammar at each floor of two different buildings:

```
Gemma (18-floor building):        BLOOM (30-floor building):

Floor 18: ██░░░░ (fading)          Floor 30: ██░░░░ (fading)
Floor 16: ████████ (loud!)         Floor 27: ████████ (loud!)
Floor 14: ███████ (building)       Floor 24: ███████ (building)
Floor 12: █████ (growing)          Floor 20: █████ (growing)
Floor 10: ███ (starting)           Floor 15: ███ (starting)
Floor  5: █ (quiet)                Floor  8: █ (quiet)
Floor  1: ░ (silent)               Floor  1: ░ (silent)
```

The buildings are different heights, but the *pattern* is the same: quiet at the bottom, building in the middle, loudest near the top, then fading at the very end. When we normalize both to relative depth (0% = ground floor, 100% = top floor), the curves overlap.

**Pearson r = 0.988** means these two curves are almost perfectly correlated — when one goes up, the other goes up by the same amount. The models are doing the same thing at the same relative stage of processing.

#### Why Magnitude Works But Accuracy Doesn't

Signal magnitude asks: "How big is the total difference between singular and plural?" This is robust — even if the difference is spread across many directions, the total size is captured.

Reading vector accuracy asks: "Can we find ONE direction that separates singular from plural?" This requires PCA to find a clean direction, which needs many examples (we only used 128) and works best when the signal is concentrated in one dimension. At most layers, the number signal is spread across many directions, so PCA struggles to isolate it.

```
Layer where signal is concentrated (like L13H7's output):
  PCA finds it easily:  ←── singular ──|── plural ──→  (high accuracy)

Layer where signal is distributed across many directions:
  PCA finds only part:  ←── mixed ──|── mixed ──→     (low accuracy)
  But total magnitude is still large!                  (high magnitude)
```

This is why magnitude is the better metric for comparing flow topology across models.

### Understanding CKA: How Can We Compare Representations of Different Sizes?

Gemma has 2048-dimensional representations. BLOOM has 2560-dimensional representations. How can we compare them? We can't just subtract one from the other — they live in different spaces.

**CKA (Centered Kernel Alignment)** solves this by ignoring the actual vectors and instead comparing the **geometry** — the pattern of which examples are similar to which.

#### The Intuition: Similarity of Similarities

Imagine you give 5 sentences to both models. At some layer, each model has a 5-point representation:

```
Sentences:  A="The doctor is"  B="The doctors are"  C="The cat is"  D="The cats are"  E="The king is"

Gemma Layer 13 (2048-dim):         BLOOM Layer 22 (2560-dim):
A and C are close (both singular)   A and C are close (both singular)
B and D are close (both plural)     B and D are close (both plural)
A and B are far apart               A and B are far apart
```

CKA computes a similarity matrix for each model — "how similar is sentence i to sentence j?" — and then measures how correlated these two similarity matrices are.

```
Gemma's similarity matrix:     BLOOM's similarity matrix:
     A    B    C    D    E          A    B    C    D    E
A  [1.0  0.2  0.8  0.1  0.7]   A  [1.0  0.3  0.9  0.2  0.8]
B  [0.2  1.0  0.1  0.9  0.2]   B  [0.3  1.0  0.2  0.8  0.1]
C  [0.8  0.1  1.0  0.2  0.6]   C  [0.9  0.2  1.0  0.1  0.7]
D  [0.1  0.9  0.2  1.0  0.3]   D  [0.2  0.8  0.1  1.0  0.2]
E  [0.7  0.2  0.6  0.3  1.0]   E  [0.8  0.1  0.7  0.2  1.0]
                                                              
These look very similar! → CKA ≈ 0.95
```

**CKA = 1.0** means the models organize information identically — same sentences are similar, same sentences are different. **CKA = 0** means no relationship.

#### Key Property: Invariance

CKA doesn't care about:
- **Dimension**: 2048 vs 2560 — doesn't matter, we only use pairwise similarities
- **Rotation**: If BLOOM's singular/plural direction points north and Gemma's points east, CKA still sees they separate the same way
- **Scaling**: If one model uses values 10x larger than the other, CKA is unaffected

This makes it perfect for cross-model comparison.

#### The Cross-Model CKA Heatmap

When we compute CKA between *every* Gemma layer and *every* BLOOM layer, we get an 18×30 heatmap:

```
         BLOOM layers →
       0    10    20    30
G  0   ■■■   .     .     .     ← Early layers match early layers
e  6   .    ■■■    .     .     ← Middle matches middle
m 12   .     .   ■■■    .     ← Critical layers align
m 18   .     .     .   ■■■    ← Output matches output
a

A clear diagonal = the models process in the same order.
A scattered pattern = different processing strategies.
```

Our results show a strong diagonal, meaning Gemma and BLOOM process SVA in the same sequence of computational stages.

### Cross-Model Results

The experiments are complete. Here are the key findings from comparing Gemma 2B and BLOOM 3B.

In [ ]:
# Cross-model results — actual data from completed experiments
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

print('=== Key Head Positions ===')
print('Gemma 2B:  L13H7  at 72% depth (layer 13 of 18)')
print('BLOOM 3B:  L23H15 at 77% depth (layer 23 of 30)')
print('→ Both models place their key SVA head at ~3/4 depth\n')

print('=== Signal Magnitude Correlation ===')
print('English:  Pearson r = 0.983')
print('Spanish:  Pearson r = 0.994')
print('→ Near-perfect correlation: both models amplify the number signal identically\n')

print('=== Cross-Model CKA ===')
print('Depth   Gemma→BLOOM   CKA')
print('  0%    L0  → L0     0.970')
print(' 50%    L9  → L7     0.940')
print(' 72%    L12 → L16    0.872')
print(' 85%    L14 → L24    0.733')
print('100%    L17 → L29    0.566')
print('→ Strong alignment through 85% of the network')

In [ ]:
# Figure: Signal shape comparison
display(Image('results/figures/fig_cross_model_signal_shape.png'))

The signal magnitude curves above show normalized signal strength at each relative depth. Despite Gemma having 18 layers and BLOOM having 30, the curves overlap almost perfectly. Both models:

1. Start with minimal signal near the input
2. Build the signal gradually through mid-layers
3. Peak near the top of the network
4. Maintain the signal to the output

This is the strongest evidence for a shared computational topology.

In [ ]:
# Figure: Cross-model CKA heatmap
display(Image('results/figures/fig_cross_model_cka_heatmap.png'))

The CKA heatmap shows which layers in Gemma build similar representations to which layers in BLOOM. The bright band along the diagonal confirms the models process information in the same order. Key observations:

- **Early layers (CKA ~0.97)**: Nearly identical representations — both models start from similar token embeddings
- **Middle layers (CKA ~0.87-0.94)**: Still strongly aligned — the models build similar intermediate representations
- **Late layers (CKA ~0.57-0.73)**: Divergence — the models specialize their output representations differently

The band bends slightly above the diagonal, meaning BLOOM's later layers (which it has more of) correspond to Gemma's middle layers. This makes sense: BLOOM has 12 extra layers, so it can spread the same computation over more steps.

In [ ]:
# Figure: Patching comparison side by side
display(Image('results/figures/fig_cross_model_patching.png'))

In [ ]:
# Figure: Key head position comparison
display(Image('results/figures/fig_cross_model_key_head.png'))

In [ ]:
# Figure: BLOOM steering results
display(Image('results/figures/fig_bloom_steering.png'))

# Part 4: Putting It All Together

## The Full Circuit Story

Here's how Gemma 2B solves subject-verb agreement, as revealed by our 8 experiments:

```
INPUT: "The doctor that helped the teacher ___"

LAYER 1: L1H7 reads the subject position (Experiment: Attention Patterns)
    ↓ writes subject identity to residual stream

LAYERS 2-12: background processing, gradual buildup

LAYER 13: THE CRITICAL LAYER
    ├── L13H7 reads the subject, encodes singular/plural as a single direction
    │   (Experiments: Patching=#1, DLA=positive, PCA=PC1 separates sg/pl)
    │   This direction is LANGUAGE-INDEPENDENT (Experiment: CKA peaks here)
    │
    ├── MLP13 neuron 2069 amplifies the number signal
    │   (Experiment: Neuron Analysis)
    │
    └── Representations CONVERGE across languages at this layer
        (Experiment: CKA = 0.267, the maximum)

LAYERS 14-16: signal routing
    ├── English: L13→L17 pathway (Experiment: Weight Connections)
    └── Other langs: L14→L16 pathway

LAYER 17: THE OUTPUT LAYER
    ├── L17H4 receives the number signal and pushes toward correct verb
    │   (Experiments: Patching=#2, DLA=positive)
    │
    ├── MLP17 is the MOST IMPORTANT component overall
    │   (Experiment: EAP — MLP17 is #1 in every language)
    │   Neuron 1138 is a key contributor (Experiment: Neuron Analysis)
    │
    └── Signal is converted to final logit difference

OUTPUT: logit("is") > logit("are") → model predicts "is" ✓
```

## The Cross-Lingual Claim

The paper's core claim — and what our extension to 7 languages supports — is that this circuit is **shared across languages**:

1. **Same heads**: L13H7 is causally important in EN, ES, RU, SW (Patching)
2. **Same direction**: PC1 from English separates sg/pl in other languages too (PCA + Steering)
3. **Same layer**: Representations converge at Layer 13 across all 21 language pairs (CKA)
4. **Same MLPs**: MLP17 dominates in every single language (EAP)

The model didn't develop separate circuits for each language. It has **one circuit** with language-specific routing paths.

## What's Novel in Our Extension (vs. the Original Paper)

| Contribution | What the paper did | What we added |
|-------------|-------------------|---------------|
| Languages | 2 (EN, ES) | 7 (+ FR, RU, TR, SW, QU) |
| EAP | Not used | Revealed MLP dominance |
| Weight analysis | Not done | Per-weight importance maps, cross-layer connections |
| CKA convergence | Not done | Layer 13 peak across 21 pairs |
| Circuit knockout | Not done | Necessity/sufficiency validation |
| Logit lens | Not done | Layer-by-layer prediction formation |
| Attention patterns | Not done | L1H7 as universal subject-reader |

---

## Summary of Methods

| Method | Type | Needs Data? | What It Measures |
|--------|------|------------|-----------------|
| Patching | Causal | Yes | What carries the signal |
| DLA | Correlational | Yes | What pushes the output |
| Neuron DLA | Correlational | Yes | Which neurons push the output |
| PCA | Statistical | Yes | Representation structure |
| Steering | Causal | Yes | Cross-lingual transfer |
| EAP | Gradient-based | Yes | Fast importance estimation |
| Weight analysis | Structural | **No** | Weight alignment with task |
| CKA | Statistical | Yes | Cross-lingual similarity |

## The Cross-Model Question (NEW)

Experiments 1-8 established that Gemma 2B uses a shared circuit across 7 languages. But is this circuit unique to Gemma, or is it a universal computational strategy?

Experiment 9 (RepE) traces the signal's emergence across layers — giving us a "route map" of how the number signal travels through the network. Experiment 10 compares this route map between Gemma 2B and BLOOM 3B (a structurally different multilingual model).

If both models show the same signal emergence profile when normalized to relative depth — the number signal appears at ~60-70% depth, peaks at ~72%, and is maintained to the output — this suggests the **computational topology for grammatical agreement is architecture-independent**, much like how visual processing follows the same stages in human and monkey brains despite different brain structures.

### What We're Testing
| Hypothesis | Test | Evidence For | Evidence Against |
|-----------|------|-------------|-----------------|
| H1: Same emergence shape | Pearson r of profiles | r > 0.7 | r < 0.3 |
| H2: Same peak depth | Peak position comparison | Δ < 10% | Δ > 20% |
| H3: Same convergence | CKA at critical layer | CKA peaks at both critical layers | CKA peaks elsewhere |
| H4: Reading vectors align | Cross-model Procrustes | Low distance | High distance |



# Appendix: Glossary of Key Terms

| Term | Meaning |
|------|---------|
| **Residual stream** | The 2048-dim vector at each position that gets passed through the entire model. Every component reads from it and adds to it. |
| **d_model (2048)** | The dimensionality of the residual stream — the main "highway" of the model. |
| **d_head (256)** | The dimensionality of each attention head's internal subspace. The head reads 2048→256 (via W_V), processes in 256, and writes 256→2048 (via W_O). |
| **d_mlp (16384)** | The number of neurons in each MLP layer. Much wider than d_model — the MLP is the "thinking" space. |
| **hook_z** | TransformerLens name for the per-head output tensor. Shape (batch, seq, n_heads, d_head). This is BEFORE the W_O projection — it's still in the head's 256-dim space. |
| **W_U (unembedding)** | The matrix that converts the final residual stream vector to logits over the vocabulary. Shape (2048, 256128). |
| **Unembed direction** | W_U[:, "is"] - W_U[:, "are"] — the single vector that determines whether the model predicts "is" or "are". All of DLA and weight analysis revolves around this vector. |
| **Logit diff** | logit("is") - logit("are"). Positive = model prefers correct verb. This is the scalar we're trying to understand. |
| **OV circuit** | W_V @ W_O — the head's "what to write" function. Maps d_model → d_model through a d_head bottleneck. |
| **QK circuit** | W_Q @ W_K^T — the head's "where to look" function. Determines attention patterns. |
| **Contrastive pair** | Two sentences differing only in subject number (sg vs pl). The experimental control. |
| **Denoising / Activation patching** | Replace one component's corrupted activation with the clean version. Measures causal importance. |
| **Normalized patch effect** | (patched - corrupted) / (clean - corrupted). Maps patch effect to [0,1]. |
| **DLA** | Direct Logit Attribution — component_output @ unembed_dir. Exact decomposition of logit diff. |
| **EAP** | Edge Attribution Patching — gradient-based approximation of patching. First-order Taylor expansion. |
| **CKA** | Centered Kernel Alignment — measures similarity of representation geometry between two sets of vectors. |
| **PC1** | First principal component — the direction of maximum variance in the data. In our case, the "number direction" in L13H7's output space. |
| **Steering** | Adding α × PC1 to a head's output during inference. A causal intervention that tests whether a direction controls behavior. |
| **Reading vector** | The direction (from RepE) that best separates contrastive activations at a given layer. Extracted via PCA on difference vectors. |
| **Signal emergence profile** | A curve showing how the number signal strength varies across layers. The model's "computational timeline." |
| **Relative depth** | Layer position normalized to [0, 1]. Allows comparison across models with different layer counts. Layer 13 in 18-layer Gemma = 72% depth. |
| **Cross-model CKA** | CKA between representations of two different models at each layer pair. Produces a (n_layers_A × n_layers_B) alignment heatmap. |
| **Difference vector** | clean_activation - corrupted_activation. Isolates the concept-specific signal by canceling shared information. |
| **ALiBi** | Attention with Linear Biases — BLOOM's positional encoding, which adds a learned linear bias to attention scores instead of using positional embeddings (RoPE in Gemma). |
